In [ ]:
pip install pandas_ta

In [ ]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas_ta as ta

In [ ]:
pip install torch_geometric

In [ ]:
# Install and import PyG modules separately; user must have torch-geometric installed.
try:
    from torch_geometric.nn import GCNConv
except Exception as e:
    raise ImportError("Please install torch-geometric following the official instructions. Error: " + str(e))

# Transformers FinBERT
from transformers import pipeline

# yfinance for price data
import yfinance as yf

### Apple

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Apple")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### HSBC

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info_part2.xlsx", sheet_name="HSBC")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Pepsi

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info_part2.xlsx", sheet_name="Pepsi")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Toyota

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Toyota")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

### Tencent

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

apple_data = pd.read_excel(r"/content/gdrive/My Drive/Dataset/News_Info.xlsx", sheet_name="Tencent")
apple_data['date']=pd.to_datetime(apple_data['date']).dt.date

apple_data.dropna(inplace=True)
apple_data

In [ ]:
import math
import random
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## ST-GCN

In [ ]:
# -------------------------
# Utilities
# -------------------------
def returns_from_prices(prices: np.ndarray) -> np.ndarray:
    """Compute log returns. prices shape (T, N) -> returns (T-1, N)"""
    return np.log(prices[1:] / prices[:-1] + 1e-12)


def finbert_sentiment_pipeline(model_name: str = "yiyanghkust/finbert-tone", device: int = -1):
    """
    Create a transformers pipeline for FinBERT sentiment analysis.
    device=-1 uses CPU; >=0 uses that GPU id.
    Returns a pipeline object that outputs labels like POS/NEG/NEU depending on model.
    """
    # Use sentiment-analysis pipeline
    pipe = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name, device=device)
    return pipe


def classify_news_sentiment(news_df: pd.DataFrame, pipe) -> pd.DataFrame:
    """
    Run FinBERT sentiment on each news text and return the news_df with 'label' and 'score' columns.
    label usually one of: 'positive', 'neutral', 'negative' (model dependent); we will normalize later.
    """
    texts = news_df["title"].tolist()
    # pipe can process batches; to be safe, process in small batches
    batch_size = 16
    labels = []
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        outs = pipe(batch)
        # outs: list of {'label': 'positive', 'score': 0.98} etc.
        for o in outs:
            labels.append(o["label"])
            scores.append(float(o.get("score", 0.0)))
    df = news_df.copy().reset_index(drop=True)
    df["label"] = labels
    df["score"] = scores
    return df


def aggregate_sentiment_per_company_day(news_df_with_labels: pd.DataFrame, tickers: List[str], date_index: pd.DatetimeIndex):
    """
    For each (date, ticker) compute an aggregated sentiment score.
    Strategy:
      - Map labels to numeric: positive -> +1, neutral -> 0, negative -> -1
      - Use label * score as weighted value, average across news mentioning the ticker (normalized format) on that date.
    Returns:
      sentiment_df: DataFrame indexed by date_index (dates from price_df) with columns tickers (shape T x N)
    """

    def label_to_num(lab):
        if pd.isna(lab):
            return 0.0
        s = str(lab).lower()
        if "pos" in s:
            return 1.0
        elif "neg" in s:
            return -1.0
        else:
            return 0.0

    tmp = news_df_with_labels.copy()
    #tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()

    # Ensure mentions are parsed lists
    if isinstance(tmp["mentions"].iloc[0], str):
        tmp["mentions"] = tmp["mentions"].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Normalize tickers in 'mentions' — e.g. "AAPL.US" -> "AAPL"
    def normalize_ticker(name):
        if not isinstance(name, str):
            return name
        return name.split(".")[0].upper().strip()

    tmp["mentions"] = tmp["mentions"].apply(lambda lst: [normalize_ticker(x) for x in lst])

    # Aggregate sentiment values
    accum = {}
    for _, row in tmp.iterrows():
        d = row["date"]
        numeric = label_to_num(row["label"]) * float(row["score"])
        for t in row["mentions"]:
            if t in tickers:  # only keep known companies
                accum.setdefault((d, t), []).append(numeric)

    # Build sentiment matrix
    rows = []
    for d in date_index:
        row_vals = []
        for t in tickers:
            vals = accum.get((d, t), [])
            row_vals.append(float(np.mean(vals)) if vals else 0.0)
        rows.append(row_vals)

    sentiment_df = pd.DataFrame(rows, index=date_index, columns=tickers).astype(np.float32)
    return sentiment_df


# -------------------------
# Graph construction from news co-occurrence
# -------------------------

def normalize_ticker(name):
    if not isinstance(name, str):
        return name
    return name.split(".")[0].upper().strip()


def news_cooccurrence_graph_for_day(
    news_df: pd.DataFrame,
    tickers: List[str],
    target_date: pd.Timestamp,
    window: int = 5,
    add_self_loops: bool = True,
    min_edge_weight: float = 1e-6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a normalized co-occurrence graph over a rolling window ending at target_date.

    Parameters:
        news_df: DataFrame with 'date' (datetime) and 'mentions' (list of tickers).
        tickers: List of valid ticker symbols (e.g., ["AAPL", "TSLA"]).
        target_date: The date for which we build the graph (use window days up to and including this date).
        window: Number of days (including target_date) to include in the co-occurrence window.
        add_self_loops: Whether to add self-loops after co-occurrence counting.
        min_edge_weight: Threshold to drop near-zero edges after normalization.

    Returns:
        edge_index: (2, E) numpy array of dtype int64.
        edge_weight: (E,) numpy array of dtype float32.
    """
    # Convert target_date to date object
    end = pd.to_datetime(target_date).date()
    start = end - pd.Timedelta(days=window - 1)

    # Filter news in the time window
    news_df['date_parsed'] = pd.to_datetime(news_df['date']).dt.date
    sub = news_df[(news_df['date_parsed'] >= start) & (news_df['date_parsed'] <= end)].copy()

    if sub.empty:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    n = len(tickers)
    idx_map = {t: i for i, t in enumerate(tickers)}

    # Initialize co-occurrence matrix
    mat = np.zeros((n, n), dtype=np.float32)

    for _, row in sub.iterrows():
        mentions = row["mentions"]
        if not isinstance(mentions, (list, tuple)):
            continue
        # Normalize and filter tickers
        valid_mentions = [normalize_ticker(m) for m in mentions if normalize_ticker(m) in idx_map]
        unique_mentions = sorted(set(valid_mentions))
        # Count pairwise co-occurrences (including self? not yet)
        for i in range(len(unique_mentions)):
            for j in range(i, len(unique_mentions)):  # include i==j for self if desired later
                a = idx_map[unique_mentions[i]]
                b = idx_map[unique_mentions[j]]
                if i == j:
                    # We'll add self-loops uniformly later; skip intra-article self-counts
                    continue
                else:
                    mat[a, b] += 1.0
                    mat[b, a] += 1.0

    # Add self-loops explicitly (standard practice in GCNs)
    if add_self_loops:
        np.fill_diagonal(mat, 1.0)  # You can also use degree or constant; 1.0 is common

    # Symmetric normalization: A_hat = D^{-1/2} A D^{-1/2}
    deg = np.sum(mat, axis=1)  # degree vector
    deg_inv_sqrt = np.where(deg > 0, deg ** (-0.5), 0.0)
    deg_inv_sqrt_matrix = np.diag(deg_inv_sqrt)
    normalized_adj = deg_inv_sqrt_matrix @ mat @ deg_inv_sqrt_matrix

    # Apply threshold to avoid storing negligible edges
    mask = normalized_adj >= min_edge_weight
    if not np.any(mask):
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    # Extract upper triangle (including diagonal) to avoid duplicates
    # But since we want undirected graph, we'll get all non-zero entries and dedup via set or just use full
    rows, cols = np.nonzero(mask)
    edges = np.stack([rows, cols], axis=0).astype(np.int64)
    weights = normalized_adj[rows, cols].astype(np.float32)

    return edges, weights


# -------------------------
# Revised DynamicGraphDataset that uses news sentiment and co-occurrence edges
# -------------------------




# revised code by adding min-max normalization
class DynamicGraphDatasetNews(torch.utils.data.Dataset):
    def __init__(
        self,
        price_df: pd.DataFrame,
        news_df: pd.DataFrame,
        sentiment_df: pd.DataFrame,
        seq_len: int = 5,
        co_window: int = 5,
        mode: str = "regression",
    ):
        assert mode in ("regression", "classification")
        self.seq_len = seq_len
        self.co_window = co_window
        self.mode = mode

        prices = price_df.values.astype(np.float32)
        dates = pd.to_datetime(price_df.index)
        self.date_index = dates
        self.tickers = list(price_df.columns)
        T, N = prices.shape

        sentiment_df = sentiment_df.reindex(dates).fillna(0.0)
        svals = sentiment_df.values.astype(np.float32)

        # Normalize prices
        norm_prices = np.zeros_like(prices, dtype=np.float32)

        for t in range(len(prices)):
            past = prices[:t+1]  # ONLY past and current

            mean_t = past.mean(axis=0)
            std_t = past.std(axis=0) + 1e-6

            norm_prices[t] = (prices[t] - mean_t) / std_t

        # ---- FEATURES ----
        feature_list = []
        for t in range(T - 1):
            feats = [
                norm_prices[t],   # (N,)
                svals[t],         # (N,)
            ]
            feat_t = np.stack(feats, axis=1)  # (N, 2)
            feature_list.append(feat_t.astype(np.float32))

        self.features = np.stack(feature_list, axis=0)  # (T-1, N, 2)
        self.targets = norm_prices[1:]  # (T-1, N)

        self.valid_end_idx = list(range(self.seq_len - 1, T - 1))

        # ---- EDGE CONSTRUCTION + NORMALIZATION ----
        self.edge_index_list = []
        self.edge_weight_list = []

        for t in range(T - 1):
            price_date = pd.to_datetime(self.date_index[t]).date()

            ei, ew = news_cooccurrence_graph_for_day(
                news_df, self.tickers,
                target_date=price_date,
                window=self.co_window
            )

            # ---- EDGE NORMALIZATION BLOCK ----
            if ew is not None and len(ew) > 0:
                ew = ew.astype(np.float32)

                # 1. Log scaling (important for co-occurrence)
                ew = np.log1p(ew)

                # 2. Min-Max normalization
                ew_min = ew.min()
                ew_max = ew.max()

                if ew_max > ew_min:
                    ew = (ew - ew_min) / (ew_max - ew_min)
                else:
                    ew = np.zeros_like(ew)

                # 3. Prevent self-loop dominance
                ew = 0.1 + 0.9 * ew

            self.edge_index_list.append(ei)
            self.edge_weight_list.append(ew)

    def __len__(self):
        return len(self.valid_end_idx)

    def __getitem__(self, idx):
        end_t = self.valid_end_idx[idx]
        start_t = end_t - (self.seq_len - 1)

        seq_feats = self.features[start_t : end_t + 1]
        seq_edge_idx = self.edge_index_list[start_t : end_t + 1]
        seq_edge_w = self.edge_weight_list[start_t : end_t + 1]
        target = self.targets[end_t]

        if self.mode == "classification":
            y = (target > 0).astype(np.int64)
        else:
            y = target.astype(np.float32)

        return {
            "seq_feats": torch.from_numpy(seq_feats),
            "seq_edge_index": seq_edge_idx,
            "seq_edge_weight": seq_edge_w,
            "target": torch.from_numpy(y),
        }
        return sample

# Collate function (reuse)
def collate_dynamic(batch):
    seq_feats = torch.stack([item["seq_feats"] for item in batch], dim=0)
    targets = torch.stack([item["target"] for item in batch], dim=0)
    seq_edge_index = [item["seq_edge_index"] for item in batch]
    seq_edge_weight = [item["seq_edge_weight"] for item in batch]
    return {"seq_feats": seq_feats, "seq_edge_index": seq_edge_index, "seq_edge_weight": seq_edge_weight, "target": targets}


# -------------------------
# ST-GCN model
# -------------------------
class STGCN(nn.Module):
    def __init__(
        self,
        in_feats: int,
        gcn_hidden: int = 64,
        tcn_hidden: int = 64,
        out_dim: int = 1,
        kernel_size: int = 5,
        dropout: float = 0.2,
        mode: str = "regression",
    ):
        super().__init__()
        self.mode = mode

        # Spatial graph convolution
        self.gcn1 = GCNConv(in_feats, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        # Temporal convolution (Corrected for Conv1d)
        self.tcn = nn.Sequential(
            nn.Conv1d(
                in_channels=gcn_hidden,
                out_channels=tcn_hidden,
                kernel_size=kernel_size,
                padding=0,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.mlp = nn.Sequential(
            nn.Linear(tcn_hidden, tcn_hidden // 2),
            nn.ReLU(),
            nn.Linear(tcn_hidden // 2, out_dim),
        )

    def forward(self, seq_feats, seq_edge_index, seq_edge_weight):
        batch, seq_len, N, F_dim = seq_feats.shape
        print(f"Input Shape: {seq_feats.shape}") # (Batch, Time, Nodes, Feats)

        gcn_outputs = []
        # 1. Spatial Phase
        for t in range(seq_len):
            batch_node_embeds = []
            for b in range(batch):
                x = seq_feats[b, t]
                ei = torch.from_numpy(seq_edge_index[b][t]).long().to(x.device) if seq_edge_index[b][t].size > 0 else torch.empty((2,0)).long().to(x.device)
                ew = torch.from_numpy(seq_edge_weight[b][t]).float().to(x.device) if seq_edge_weight[b][t].size > 0 else None

                h = torch.relu(self.gcn1(x, ei, ew))
                h = torch.relu(self.gcn2(h, ei, ew))
                batch_node_embeds.append(h)
            gcn_outputs.append(torch.stack(batch_node_embeds, dim=0))

        x = torch.stack(gcn_outputs, dim=1)
       #print(f"After GCN Phase: {x.shape}") # (Batch, Time, Nodes, GCN_Hidden)

        # 2. Temporal Phase
        x = x.permute(0, 2, 3, 1).contiguous()
        #print(f"Permuted for TCN: {x.shape}") # (Batch, Nodes, GCN_Hidden, Time)

        x = x.view(batch * N, -1, seq_len)
        #print(f"Flattened for Conv1d: {x.shape}") # (Batch*Nodes, GCN_Hidden, Time)

        x = self.tcn(x)
        #print(f"After TCN (Conv1d): {x.shape}") # (Batch*Nodes, TCN_Hidden, Time)

        x = x[:, :, -1]
        #print(f"Last Time Step: {x.shape}") # (Batch*Nodes, TCN_Hidden)

        x = x.view(batch, N, -1)
        #print(f"Reshaped for MLP: {x.shape}") # (Batch, Nodes, TCN_Hidden)

        # 3. Prediction
        preds = self.mlp(x)
        #print(f"Final Output Shape: {preds.shape}") # (Batch, Nodes, Out_Dim)

        return preds.squeeze(-1)




def full_news_demo(
    tickers: List[str] = None,
    start_date: str = apple_data["date"].min(),
    end_date: str = apple_data["date"].max(),
    n_news: int = len(apple_data),
    seq_len: int = 5,
    co_window: int = 5,
    batch_size: int = 16,
    epochs: int = 6,
    device_str: str = "cpu",
):
    device = torch.device(device_str)
    if tickers is None:
        # default small set (pick companies with yfinance tickers)
        tickers = ["AAPL", "AMZN", "GOOGL", "TSLA", "NFLX", "MSFT", "META", "005930.KS", "CMCSA", "IT"]

    print("Downloading price data with yfinance...")
    price_df = yf.download(tickers, start=start_date, end=end_date, progress=False)["Close"]
    # yfinance returns multi-column if multiple tickers; ensure DataFrame shape (T, N)
    if isinstance(price_df.columns, pd.MultiIndex):
        # some tickers may be missing; flatten or select 'Adj Close' level
        price_df = price_df.copy()
    price_df = price_df.dropna(how="all").ffill().dropna(axis=1)  # drop tickers with no data
    # ensure tickers order aligns
    price_df = price_df[tickers]
    print(f"Price data shape: {price_df.shape}")

    # Generate synthetic news
    print("Generating synthetic news...")
    news_df = apple_data[['date','title', 'symbols']] # Include symbols for mentions
    news_df.rename(columns={'symbols': 'mentions'}, inplace=True)
    # run FinBERT sentiment classification (this will download model the first time)
    print("Loading FinBERT pipeline (this may take a while for first run)...")
    pipe = finbert_sentiment_pipeline(model_name="yiyanghkust/finbert-tone", device=-1)  # cpu; set device=0 for GPU
    print("Classifying synthetic news with FinBERT...")
    news_labeled = classify_news_sentiment(news_df, pipe)
    news_labeled.reset_index(inplace=True, drop=True) # Reset index to make date a column again
    print("Sample labeled news:")
    print(news_labeled.head())

    # Aggregate sentiment per date-company
    print("Aggregating sentiment per company-day...")
    # create date_index for price_df
    date_index = price_df.index.date
    sentiment_df = aggregate_sentiment_per_company_day(news_labeled, list(price_df.columns), date_index)
    print("Sentiment df head:")
    print(sentiment_df.head())

    # build dataset
    print("Building DynamicGraphDatasetNews...")
    dataset = DynamicGraphDatasetNews(price_df=price_df, news_df=news_labeled, sentiment_df=sentiment_df, seq_len=seq_len, co_window=co_window, mode="regression")
    n = len(dataset)
    train_n = int(0.7 * n)
    val_n = int(0.15 * n)
    idxs = list(range(n))
    train_idx = idxs[:train_n]
    val_idx = idxs[train_n : train_n + val_n]
    test_idx = idxs[train_n + val_n :]

    from torch.utils.data import Subset, DataLoader

    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, collate_fn=collate_dynamic)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_dynamic)

    in_feats = dataset.features.shape[-1]
    print(f"in_feats={in_feats}, dataset length={len(dataset)}")

    model = STGCN(
    in_feats=in_feats,
    gcn_hidden=64,
    tcn_hidden=64,
    out_dim=1,
    kernel_size=5,
    dropout=0.2,
    mode="regression"
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, device, loss_fn)
        val_loss, val_metrics, val_pred, val_targ = eval_epoch(model, val_loader, device, loss_fn, mode="regression")
        print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val Metrics: {val_metrics}")

    test_loss, test_metrics, preds, targ = eval_epoch(model, test_loader, device, loss_fn, mode="regression")
    print(f"Test Loss: {test_loss:.6f} | Test Metrics: {test_metrics}")

    return model, dataset, price_df, news_labeled, sentiment_df, preds, targ

### Apple

In [ ]:
if __name__ == "__main__":
    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ = full_news_demo(
        tickers = ["AAPL","MSFT","GOOGL","AMZN","META","NVDA","INTC","AMD","QCOM","AVGO",
        "TSM","ASML","TXN","MU","IBM","ORCL","ADBE","CRM","CSCO","HPQ",
        "DELL","SONY","005930.KS","000660.KS","V","MA","PYPL","SHOP","NFLX","DIS","WMT","COST","T","VZ","TMUS","BIDU","JD",
        "BABA","NTES","0700.HK","TSLA","F","GM","STX","WDC","LRCX"],
        #tickers=["0005.HK","JPM","2888.HK","MS","UBS","C","GS","NWG","AAPL","MSFT"],   #for HSBC
        #tickers = ["PEP","KO", "WMT", "COST", "CL", "CVX", "PG", "JNJ", "AAPL", "MSFT", "AMZN", "TSLA", "IBM"], #for Pepsi
        #tickers = ["7203.T","7201.T", "7267.T", "7261.T", "F", "GM", "VOW3.DE", "BMW.DE", "TSLA", "005380.KS", "002594.SZ", "AAPL", "AMZN"],
        #tickers = ["0700.HK", "NTES", "BABA", "JD", "BIDU", "MSFT", "AAPL", "GOOGL", "AMZN", "META", "NVDA", "INTC"],
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=5,
        device_str="cpu",
    )

In [ ]:
import matplotlib.pyplot as plt

# Assume preds and targ have shape (num_samples, num_companies)
company_idx = 0  # Apple
plt.figure(figsize=(10, 5))
plt.plot(targ[:, company_idx], label='True')
plt.plot(preds[:, company_idx], label='Predicted')
plt.title(f"Predicted vs True Prices for Apple")
plt.xlabel("Sample")
plt.ylabel("Price")
plt.legend()
plt.show()

### HSBC

In [ ]:
if __name__ == "__main__":
    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ = full_news_demo(
        tickers = [
        "HSBC","JPM","BAC","C","GS","MS","WFC","SCHW","AXP","BLK",
        "SPGI","ICE","CME","UBS","BARC.L","LLOY.L","NWG","STAN.L","DB",
        "BNP.PA","SAN.PA","BBVA","ING","SMFG","MUFG","MFG","HDB","IBN",
        "RY","TD","BNS","CM","PNC","USB","TFC","AIG","MET","PRU",
        "V","MA","PYPL","AAPL","MSFT","AMZN","GOOGL","BABA","0700.HK","JD"
        ],
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=5,
        device_str="cpu",
    )

In [ ]:
import matplotlib.pyplot as plt

# Assume preds and targ have shape (num_samples, num_companies)
company_idx = 0  # Apple
plt.figure(figsize=(10, 5))
plt.plot(targ[:, company_idx], label='True')
plt.plot(preds[:, company_idx], label='Predicted')
plt.title(f"Predicted vs True Prices for HSBC")
plt.xlabel("Sample")
plt.ylabel("Price")
plt.legend()
plt.show()

### Pepsi

In [ ]:
if __name__ == "__main__":
    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ = full_news_demo(
        tickers = [
        "PEP","KO","MNST","KDP","STZ","BF-B","PG","CL","UL","NSRGY",
        "MDLZ","GIS","HSY","SJM","KHC","CPB","CAG","ADM","BG",
        "WMT","COST","TGT","KR","SYY","MCD","YUM","CMG","DPZ","SBUX",
        "NKE","AAPL","MSFT","AMZN","TSLA","NVDA","DIS","NFLX","META","GOOGL",
        "IBM","INTC","CVX","XOM","BP","PFE","JNJ","MRK","ABT","ABBV"
        ],
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=5,
        device_str="cpu",
    )

In [ ]:
import matplotlib.pyplot as plt

# Assume preds and targ have shape (num_samples, num_companies)
company_idx = 0  # Apple
plt.figure(figsize=(10, 5))
plt.plot(targ[:, company_idx], label='True')
plt.plot(preds[:, company_idx], label='Predicted')
plt.title(f"Predicted vs True Prices for Pepsi")
plt.xlabel("Sample")
plt.ylabel("Price")
plt.legend()
plt.show()

### Toyota

In [ ]:
if __name__ == "__main__":
    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ = full_news_demo(
        tickers = [
        "7203.T","7201.T","7267.T","7261.T","7269.T","6902.T","7202.T","7270.T","7272.T",
        "TSLA","F","GM","VOW3.DE","BMW.DE",
        "MBG.DE","STLA","RACE","HMC","005380.KS","000270.KS","1211.HK","0175.HK","2333.HK",
        "BYDDF","BYDDY","AAPL","NVDA","INTC","AMD","QCOM","TSM","7269.T",
        "TM","GM","F","AAPL","AMZN","MSFT","GOOGL","META","BABA","JD"
        ],
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=5,
        device_str="cpu",
    )

In [ ]:
# Assume preds and targ have shape (num_samples, num_companies)
company_idx = 0  # Apple
plt.figure(figsize=(10, 5))
plt.plot(targ[:, company_idx], label='True')
plt.plot(preds[:, company_idx], label='Predicted')
plt.title(f"Predicted vs True Prices for Toyota")
plt.xlabel("Sample")
plt.ylabel("Price")
plt.legend()
plt.show()

### Tencent

In [ ]:
if __name__ == "__main__":
    # Quick run (may download models & price data)
    model, dataset, price_df, news_labeled, sentiment_df, preds, targ = full_news_demo(
        tickers = [
        "0700.HK","BABA","JD","BIDU","NTES","WB","KWEB",
        "META","GOOGL","AMZN","AAPL","MSFT","NVDA",
        "INTC","AMD","TSM","QCOM","CRM","ADBE","ORCL","IBM",
        "PYPL","V","MA","NFLX","DIS","EA","TTWO","SONY",
        "005930.KS","000660.KS","9984.T","6758.T"
        ],
        start_date=apple_data["date"].min(),
        end_date=apple_data["date"].max(),
        n_news=len(apple_data),
        seq_len=5,
        co_window=5,
        batch_size=8,
        epochs=5,
        device_str="cpu",
    )

In [ ]:
# Assume preds and targ have shape (num_samples, num_companies)
company_idx = 0  # Apple
plt.figure(figsize=(10, 5))
plt.plot(targ[:, company_idx], label='True')
plt.plot(preds[:, company_idx], label='Predicted')
plt.title(f"Predicted vs True Prices for Tencent")
plt.xlabel("Sample")
plt.ylabel("Price")
plt.legend()
plt.show()